# BLIP-2 LoRA fine-tune — garment attribute captioner

Fine-tunes `Salesforce/blip2-opt-2.7b` with LoRA to emit the six garment fields
(`subcategory, fabric, formality, neckline, sleeveLength, pattern`) as JSON. Run
on a CUDA host (Colab/RunPod). Prepare data first with `dataset_prep.py`.


In [ ]:
# 1. Environment
!pip -q install torch transformers peft accelerate bitsandbytes datasets pillow


In [ ]:
# 2. Config
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

BASE_MODEL = "Salesforce/blip2-opt-2.7b"
DATA_DIR = "./data"          # train.jsonl / val.jsonl / test.jsonl from dataset_prep.py
OUTPUT_DIR = "./adapter"     # LoRA weights land here
PROMPT = ("Describe this garment as JSON with keys subcategory, fabric, "
          "formality, neckline, sleeveLength, pattern.")


In [ ]:
# 3. Load BLIP-2 (4-bit) and attach LoRA (r=16, alpha=32, q/k/v/o_proj)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
processor = Blip2Processor.from_pretrained(BASE_MODEL)
model = Blip2ForConditionalGeneration.from_pretrained(BASE_MODEL, quantization_config=bnb, torch_dtype=torch.float16)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()


In [ ]:
# 4. Dataset — JSONL -> (image, prompt, target JSON string)
import json, os
from PIL import Image
from torch.utils.data import Dataset

class WardrobeCaptionDS(Dataset):
    def __init__(self, path):
        self.rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        return Image.open(r["image"]).convert("RGB"), r["prompt"], r["target"]

def collate(batch):
    images, prompts, targets = zip(*batch)
    enc = processor(images=list(images), text=list(prompts), return_tensors="pt", padding=True)
    labels = processor.tokenizer(list(targets), return_tensors="pt", padding=True).input_ids
    labels[labels == processor.tokenizer.pad_token_id] = -100
    enc["labels"] = labels
    return enc

train_ds = WardrobeCaptionDS(os.path.join(DATA_DIR, "train.jsonl"))
val_ds   = WardrobeCaptionDS(os.path.join(DATA_DIR, "val.jsonl"))
len(train_ds), len(val_ds)


In [ ]:
# 5. Train
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir=OUTPUT_DIR, per_device_train_batch_size=2, gradient_accumulation_steps=8,
    learning_rate=2e-4, num_train_epochs=3, fp16=True, logging_steps=25,
    eval_strategy="epoch", save_strategy="epoch", remove_unused_columns=False, report_to=[],
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=collate)
trainer.train()
model.save_pretrained(OUTPUT_DIR)


In [ ]:
# 6. Eval — per-attribute accuracy of the LoRA model vs the base model on test split
import json
from attributes import TARGET_FIELDS, parse_model_json   # reuse the shared schema

def predict(m, ds_path, n=200):
    rows = [json.loads(l) for l in open(ds_path, encoding="utf-8")][:n]
    preds, golds = [], []
    for r in rows:
        img = Image.open(r["image"]).convert("RGB")
        inp = processor(images=img, text=PROMPT, return_tensors="pt").to(m.device)
        out = m.generate(**inp, max_new_tokens=128)
        preds.append(parse_model_json(processor.batch_decode(out, skip_special_tokens=True)[0]))
        golds.append(json.loads(r["target"]))
    return preds, golds

def per_attr_accuracy(preds, golds):
    acc = {}
    for fld in TARGET_FIELDS:
        hit = sum(1 for p, g in zip(preds, golds) if str(p.get(fld)) == str(g.get(fld)))
        acc[fld] = hit / max(len(golds), 1)
    return acc

test_path = os.path.join(DATA_DIR, "test.jsonl")
ft_preds, golds = predict(model, test_path)
print("fine-tuned:", per_attr_accuracy(ft_preds, golds))


In [ ]:
# 7. Push to Hub (set HF token first: notebook_login())
# from huggingface_hub import notebook_login; notebook_login()
REPO = "your-username/wardrobe-blip2-lora"
model.push_to_hub(REPO)
processor.push_to_hub(REPO)
print("Pushed. Deploy serve.py with BLIP2_ADAPTER_DIR pointing at this repo, then "
      "set BLIP2_ENDPOINT_URL in backend/.env to the deployed URL.")
